# Armenian MT: Modern Armenian -> Classical Armenian (Grabar)

Interactive demo. Run all cells, then type Armenian text in the last cell.

Model: NLLB-200 distilled-600M + LoRA (v3). chrF 51.8 / BLEU 21.6 on held-out test.


In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers peft sentencepiece torch sacrebleu
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
import os, urllib.request
ADAPTER_DIR = '/content/nllb_lora_grabar'
os.makedirs(ADAPTER_DIR, exist_ok=True)
base = 'https://raw.githubusercontent.com/SlavH/ashkarabar-to-grabar/main/checkpoints/nllb_lora_grabar'
for f in ['adapter_config.json', 'adapter_model.safetensors']:
    dest = os.path.join(ADAPTER_DIR, f)
    if not os.path.exists(dest):
        print('downloading', f)
        urllib.request.urlretrieve(f'{base}/{f}', dest)
    print(f, os.path.getsize(dest))


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = 'facebook/nllb-200-distilled-600M'
SRC_LANG = 'hye_Armn'
TGT_LANG = 'hye_Armn'
MAX_LEN = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()
print('loaded on', device)

def translate(text, num_beams=6, length_penalty=1.2, no_repeat_ngram_size=4):
    tokenizer.src_lang = SRC_LANG
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    forced_bos = tokenizer.convert_tokens_to_ids(TGT_LANG)
    with torch.no_grad():
        out = model.generate(**enc, forced_bos_token_id=forced_bos,
                             max_new_tokens=MAX_LEN, num_beams=num_beams,
                             no_repeat_ngram_size=no_repeat_ngram_size, early_stopping=True,
                             length_penalty=length_penalty)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0].strip()


In [ ]:
# Try it
examples = [
    'Աստուած ստեղծեց երկինքն ու երկիրը։',
    'Ժողովուրդը պատասխան տուեց եւ ասաց.',
    'Բարեւ, ինչպէս ես։',
]
for s in examples:
    print(f'ASH  : {s}')
    print(f'GRB  : {translate(s)}')
    print()


In [ ]:
# Interactive prompt
while True:
    text = input('ASH > ').strip()
    if not text:
        break
    print('GRB >', translate(text))
    print()
